In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from typeguard import value

# Data Loading and Visualization

In [ ]:
s1 = pd.read_pickle('../data/nanotabpfn_results.pickle.xz', compression='xz')


In [ ]:
s1.head()

In [ ]:
sorted(s1['parameters'].unique())

In [ ]:
# Loss smoothing

# I want to smooth the train loss so I can look at it and I want to smooth it in the intervall that I need.
# just take the val loss as reference.

In [ ]:
value_columns = ['real_data/nll', 'real_data/roc_auc','val/val_loss'] # , 'Loss/train'
# create a figure with as many columns as value columns
fig, axes = plt.subplots(
    nrows=1,
    ncols=len(value_columns),
    figsize=(4.5 * len(value_columns), 4),
)

for idx, row in s1.iterrows():
    for ax_idx, value_col in enumerate(value_columns):
        steps = row[value_col].index
        flops = row['total_flops'][steps].to_numpy()
        axes[ax_idx].plot(flops, row[value_col].to_numpy(), alpha=0.01, color="black")

for ax_idx, value_col in enumerate(value_columns):
    axes[ax_idx].set_title(value_col)
    
    axes[ax_idx].set_xscale("log")
    
    if "nll" in value_col.lower() or "loss" in value_col.lower():
        axes[ax_idx].set_yscale("log")

In [ ]:
# Todo: Smooth the training loss

In [ ]:
config_columns = [col for col in s1.columns if col.startswith('config')]
# drop constant columns
constant_columns = [col for col in config_columns if s1[col].nunique() == 1]
config_columns = [col for col in config_columns if col not in constant_columns]
print("config columns: ", config_columns)
loss_columns = ['real_data/nll', 'real_data/roc_auc','val/val_loss']
every_step_columns = ['total_flops', 'total_cells']
df = None
for loss_column in loss_columns:
    next_df = pd.DataFrame(pd.concat(s1[loss_column].to_list(), keys=s1.index))
    if df is None:
        df = next_df
    else:
        df = df.join(next_df, how='outer', validate='one_to_one')

for every_step_column in every_step_columns:
    next_df = pd.DataFrame(pd.concat(s1[every_step_column].to_list(), keys=s1.index))
    df = df.join(next_df, how='left', validate='one_to_one')


df = df.join(s1[config_columns + ["parameters"]], how='left', on=df.index.get_level_values(0)).drop(columns="key_0")
    

In [ ]:
df.head()

# Scaling Trends

In [ ]:
def get_pareto_frontier(
    df: pd.DataFrame,
    x_name="flops",
    y_name="Validation Loss",
    minimization=True,
) -> pd.DataFrame: 
    """ Function to compute Pareto over FLOPs.
    """
    # NOTE: strict assumption here that x_name is maximized and y_name is minimized
    df_copy = df.copy()
    if not minimization:
        df_copy[y_name] = - df_copy[y_name]
    df_sorted = df_copy.sort_values(by=[x_name, y_name], ascending=[True, True])
    df_sorted = df_sorted.drop_duplicates(subset=[x_name], keep="first")
    pareto_points = []
    min_loss_so_far = float('inf')
    for _, row in df_sorted.iterrows():
        if row[y_name] < min_loss_so_far:
            pareto_points.append(row)
            min_loss_so_far = row[y_name]
    
    df_copy = pd.DataFrame(pareto_points)
    if not minimization:
        df_copy[y_name] = - df_copy[y_name]
    return df_copy

pareto = get_pareto_frontier(df, 'total_flops', 'real_data/nll')
pareto

In [ ]:
import math
import numpy as np
values = np.array([
    1.00000000e+11, 1.77827941e+11, 3.16227766e+11, 5.62341325e+11,
    1.00000000e+12, 1.77827941e+12, 3.16227766e+12, 5.62341325e+12,
    1.00000000e+13, 1.77827941e+13, 3.16227766e+13, 5.62341325e+13,
    1.00000000e+14, 1.77827941e+14, 3.16227766e+14, 5.62341325e+14,
    1.00000000e+15, 1.77827941e+15, 3.16227766e+15, 5.62341325e+15,
    1.00000000e+16, 1.77827941e+16, 3.16227766e+16, 5.62341325e+16,
    1.00000000e+17
])

def substitute_closest(x, arr=values):
    idx = np.abs(arr - x).argmin()
    return arr[idx]
def round_sig(x, sig=2):
    if x == 0:
        return 0
    return round(x, sig - int(math.floor(math.log10(abs(x)))) - 1)


# df['total_flops_rounded'] = df['total_flops'].map(lambda x: round_sig(x, 2))
df['total_flops_rounded'] = df['total_flops'].map(substitute_closest)

flops_column = 'total_flops_rounded'

trend_columns =  config_columns + ["parameters", "total_cells"]
loss_columns = ['real_data/nll', 'real_data/roc_auc','val/val_loss']

fig, axes = plt.subplots(
    nrows=len(trend_columns)+1,
    ncols=len(loss_columns),
    figsize=(4.5 * len(loss_columns), 3.5 * (len(trend_columns)+1)),
    sharex=True,
    layout='constrained',
)

for loss_idx, loss_column in enumerate(loss_columns):
    
    maximization = "roc_auc" in loss_column or "accuracy" in loss_column
    minimization = not maximization 
    pareto = get_pareto_frontier(df, flops_column, loss_column, minimization=minimization)
    for trend_idx, trend_column in enumerate([loss_column] + trend_columns):
        ax = axes[trend_idx, loss_idx]
        if loss_idx == 0:
            ax.set_ylabel(trend_column.split('/')[-1])
        if trend_idx == 0:
            ax.set_title(loss_column)
        ax.plot(pareto[flops_column], pareto[trend_column], marker='x')
        ax.set_xscale("log")
        
        # Set the y-axis
        possible_y_values = df[trend_column].unique()
        
        if "weight_decay" in trend_column:
            ax.set_yscale("symlog", linthresh=min(possible_y_values[possible_y_values>0]))
            ax.set_ylim(-min(possible_y_values[possible_y_values>0])*.2, max(possible_y_values)*1.2)
        elif trend_idx == 0 and minimization: # adding the respective loss
            ax.set_yscale("log")
        else:    
            ax.set_yscale("log")
            ax.set_ylim(min(possible_y_values)*0.8, max(possible_y_values)*1.2)
        
        if len(possible_y_values) < 20:
            # maybe indicate it with hlines
            for y in possible_y_values:
                ax.axhline(y=y, color='lightgray', linewidth=1)
            
fig.supxlabel("FLOPs")


# Hyperparameter Importance

In [ ]:
import shap
from tabpfn import TabPFNRegressor
import numpy as np
from sklearn.ensemble import RandomForestRegressor

def plot_hyperparameter_importance(df, config_columns, metrics, flops):
    X = df[config_columns].copy()
    for col in config_columns:
        # Discretize the values...
        # X[col] = 
        pass
        
    # fit a model
    
    for metric in metrics:
        y = df[metric]
        # model = TabPFNRegressor()
        model = RandomForestRegressor(n_estimators=100, max_depth=None, random_state=42, n_jobs=-1)
        model.fit(X, y)

        explainer = shap.Explainer(model)
        shap_values = explainer(X)
        shap.plots.beeswarm(shap_values, show=False)
        plt.title("Hyperparameter Importance - " + metric + " - FLOPs: " + str(flops))
        plt.show()
        # spearman correlation

    corr = df[metrics].corr(method='spearman')

    return corr
for flops in sorted(df['total_flops_rounded'].unique()):
    print("FLOPs: ", flops)
    subset_df = df[df['total_flops_rounded'] == flops]
    plot_hyperparameter_importance(subset_df, config_columns, loss_columns, flops)

In [ ]:
# I should look how I plotted hyperparameter importance before...
# Search the cluster for that notebook
# Todo: Changes - Use TabPFN for the HP imoprtance --- Hoptentially look at HyperSHAP

plot_hyperparameter_importance(df, config_columns, loss_columns)

In [ ]:

# I want hyperparameter importance based on tabPFN for all the hyperparmeters

# For reruns: Higher learning rate, Lower hidden dimension, more layers?

In [ ]:
sorted(df['total_flops_rounded'].unique())

In [ ]:
df['total_flops_rounded'].value_counts()